# YoungXCode — Client Analytics & Retention Analysis Report
### Real Data · 50 Clients · 80 Projects · 15 Cities · 12 Industries
**Place `YoungXCode_BusinessDataset.xlsx` in the same folder, then Run All.**  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Folium


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
import warnings, os
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f9f9f9',
    'axes.grid':True,'grid.alpha':0.35,
    'axes.spines.top':False,'axes.spines.right':False,
    'font.family':'DejaVu Sans',
})
print('Libraries loaded.')


## 1. Load Real YoungXCode Data

In [ ]:
FILE = 'YoungXCode_BusinessDataset.xlsx'
if not os.path.exists(FILE):
    raise FileNotFoundError(f'Cannot find {FILE}.\nPlace the Excel file in the same folder as this notebook.')

clients  = pd.read_excel(FILE, sheet_name='Clients')
projects = pd.read_excel(FILE, sheet_name='Projects')
invoices = pd.read_excel(FILE, sheet_name='Invoices')
for d in [clients, projects, invoices]:
    d.columns = d.columns.str.strip()

clients['Join Date']            = pd.to_datetime(clients['Join Date'],   dayfirst=True, errors='coerce')
projects['Start Date']          = pd.to_datetime(projects['Start Date'], errors='coerce')
projects['Actual End Date']     = pd.to_datetime(projects['Actual End Date'], errors='coerce')
invoices['Due Date']            = pd.to_datetime(invoices['Due Date'],   errors='coerce')
invoices['Paid Date']           = pd.to_datetime(invoices['Paid Date'],  errors='coerce')
TODAY = pd.Timestamp('2025-01-15')

# Last project date per client
last_proj = projects.groupby('Client Name')['Start Date'].max().reset_index()
last_proj.columns = ['Client Name','Last Project Date']
clients = clients.merge(last_proj, on='Client Name', how='left')
clients['days_since_last'] = (TODAY - clients['Last Project Date']).dt.days
clients['is_churned']      = clients['days_since_last'] > 180

print(f'Clients: {len(clients)} | Projects: {len(projects)} | Invoices: {len(invoices)}')
print(f'Cities: {clients["City"].nunique()} | Industries: {clients["Industry"].nunique()}')
clients.head()


## 2. Retention Rate & Key KPIs

In [ ]:
total_clients    = len(clients)
repeat_clients   = (clients['Total Projects'] > 1).sum()
one_proj_clients = (clients['Total Projects'] == 1).sum()
retention_rate   = repeat_clients / total_clients * 100
churned_count    = clients['is_churned'].sum()
churn_rate       = churned_count / total_clients * 100
total_revenue    = clients['Total Revenue (INR)'].sum()
avg_clv          = clients['Total Revenue (INR)'].mean()
avg_projects     = clients['Total Projects'].mean()

print('='*56)
print('   CLIENT ANALYTICS — KEY PERFORMANCE INDICATORS')
print('='*56)
print(f'  Total Clients          : {total_clients}')
print(f'  Repeat Clients (>1 proj): {repeat_clients}')
print(f'  One-Project Clients    : {one_proj_clients}')
print(f'  Retention Rate         : {retention_rate:.1f}%   (Repeat/Total x 100)')
print(f'  Churned Clients        : {churned_count}  (no project in 6+ months)')
print(f'  Churn Rate             : {churn_rate:.1f}%')
print(f'  Total Portfolio Revenue: Rs.{total_revenue:,.0f}')
print(f'  Avg Client LTV         : Rs.{avg_clv:,.0f}')
print(f'  Avg Projects per Client: {avg_projects:.1f}')
print('='*56)


## 3. Repeat vs One-Project Client Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Client Retention & Repeat Business', fontsize=15, fontweight='bold')

axes[0].pie([repeat_clients, one_proj_clients],
    labels=[f'Repeat ({repeat_clients})', f'One-Project ({one_proj_clients})'],
    autopct='%1.1f%%', colors=['#1B5E20','#B71C1C'],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2.5))
axes[0].set_title(f'Retention Rate: {retention_rate:.1f}%', fontsize=13)

bins   = range(1, 14)
counts = clients['Total Projects'].value_counts().reindex(bins, fill_value=0)
bcols  = ['#B71C1C' if i==1 else '#1565C0' for i in bins]
bars   = axes[1].bar(list(bins), counts.values, color=bcols, edgecolor='white')
for bar, val in zip(bars, counts.values):
    if val > 0:
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                     str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Number of Projects per Client')
axes[1].set_ylabel('Number of Clients')
axes[1].set_title('Project Count Distribution', fontsize=12)
axes[1].set_xticks(list(bins))
axes[1].legend(handles=[
    mpatches.Patch(color='#B71C1C', label='One-project (at-risk)'),
    mpatches.Patch(color='#1565C0', label='Multi-project (retained)')], fontsize=9)
plt.tight_layout()
plt.savefig('retention_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Client Lifetime Value (CLV)

In [ ]:
clv = clients[['Client Name','Industry','City','Total Projects','Total Revenue (INR)','is_churned']].copy()
clv['CLV Tier'] = pd.cut(clv['Total Revenue (INR)'],
    bins=[0,2e6,4e6,6e6,8e6], labels=['Low (<2M)','Mid (2-4M)','High (4-6M)','Premium (6-8M)'])
clv = clv.sort_values('Total Revenue (INR)', ascending=False)

print('CLIENT LIFETIME VALUE TABLE (Top 20):')
print(clv.head(20).to_string(index=False))

TIER_COLORS = {'Low (<2M)':'#90A4AE','Mid (2-4M)':'#1565C0','High (4-6M)':'#F57F17','Premium (6-8M)':'#1B5E20'}
top20 = clv.head(20)
bar_cols = [TIER_COLORS[t] for t in top20['CLV Tier']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Client Lifetime Value (CLV)', fontsize=15, fontweight='bold')

bars = axes[0].barh(top20['Client Name'], top20['Total Revenue (INR)']/1e6, color=bar_cols, edgecolor='white')
for bar, val in zip(bars, top20['Total Revenue (INR)']):
    axes[0].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                 f'Rs.{val/1e6:.2f}M', va='center', fontsize=8)
axes[0].set_xlabel('Lifetime Revenue (Rs. Millions)')
axes[0].set_title('Top 20 Clients by CLV', fontsize=12)
axes[0].set_xlim(0, top20['Total Revenue (INR)'].max()/1e6 * 1.38)

tier_counts = clv['CLV Tier'].value_counts()
tc_order = [t for t in ['Premium (6-8M)','High (4-6M)','Mid (2-4M)','Low (<2M)'] if t in tier_counts.index]
axes[1].pie([tier_counts[t] for t in tc_order], labels=tc_order,
    autopct='%1.1f%%', colors=[TIER_COLORS[t] for t in tc_order],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('CLV Tier Distribution', fontsize=12)
plt.tight_layout()
plt.savefig('clv_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. New Client Acquisition Trend

In [ ]:
clients['Join Month'] = clients['Join Date'].dt.to_period('M')
monthly_acq = clients.groupby('Join Month').size().reset_index(name='new_clients')
monthly_acq['month_str']  = monthly_acq['Join Month'].astype(str)
monthly_acq['cumulative'] = monthly_acq['new_clients'].cumsum()

print('Monthly Client Acquisition:')
print(monthly_acq[['month_str','new_clients','cumulative']].to_string(index=False))

fig, ax1 = plt.subplots(figsize=(15, 5))
x = range(len(monthly_acq))
bar_cols4 = ['#1B5E20' if v >= 2 else '#1565C0' for v in monthly_acq['new_clients']]
bars = ax1.bar(x, monthly_acq['new_clients'], color=bar_cols4, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, monthly_acq['new_clients']):
    if val > 0:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(x, monthly_acq['cumulative'], color='#B71C1C', marker='o', linewidth=2, markersize=5, label='Cumulative')
ax2.set_ylabel('Cumulative Clients', color='#B71C1C', fontsize=10)
ax2.tick_params(axis='y', labelcolor='#B71C1C')
step = max(1, len(monthly_acq)//14)
ax1.set_xticks(list(x)[::step])
ax1.set_xticklabels(monthly_acq['month_str'].iloc[::step], rotation=45, ha='right', fontsize=8)
ax1.set_xlabel('Month')
ax1.set_ylabel('New Clients Acquired')
ax1.set_title('Monthly New Client Acquisition Trend (2018-2024)', fontsize=13, fontweight='bold')
ax1.legend([ax2.lines[0]], ['Cumulative'], loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('acquisition_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Client Churn Analysis

In [ ]:
churned = clients[clients['is_churned']].sort_values('Total Revenue (INR)', ascending=False)
print(f'CHURNED CLIENTS: {len(churned)}')
print(churned[['Client Name','City','Industry','Total Projects','Total Revenue (INR)','days_since_last']].to_string(index=False))

churn_city = churned.groupby('City').agg(count=('Client Name','count'), revenue=('Total Revenue (INR)','sum')).sort_values('count', ascending=False)
churn_ind  = churned.groupby('Industry').agg(count=('Client Name','count'), revenue=('Total Revenue (INR)','sum')).sort_values('count', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Client Churn Analysis', fontsize=15, fontweight='bold')

axes[0].pie([total_clients-churned_count, churned_count],
    labels=[f'Active ({total_clients-churned_count})', f'Churned ({churned_count})'],
    autopct='%1.1f%%', colors=['#2E7D32','#C62828'],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title(f'Churn Rate: {churn_rate:.1f}%', fontsize=12)

c_cols = sns.color_palette('Reds_r', len(churn_city))
bars_c = axes[1].barh(churn_city.index, churn_city['count'], color=c_cols, edgecolor='white')
for bar, val in zip(bars_c, churn_city['count']):
    axes[1].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Churned Clients')
axes[1].set_title('Churned by City', fontsize=12)

i_cols = sns.color_palette('OrRd', len(churn_ind))
bars_i = axes[2].barh(churn_ind.index, churn_ind['revenue']/1e6, color=i_cols, edgecolor='white')
for bar, val in zip(bars_i, churn_ind['revenue']):
    axes[2].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2, f'Rs.{val/1e6:.1f}M', va='center', fontsize=9)
axes[2].set_xlabel('Revenue (Rs. Millions)')
axes[2].set_title('Churned Revenue by Industry', fontsize=12)
plt.tight_layout()
plt.savefig('churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Top 10 Most Valuable Clients

In [ ]:
clients['rev_norm']    = clients['Total Revenue (INR)'] / clients['Total Revenue (INR)'].max()
clients['proj_norm']   = clients['Total Projects'] / clients['Total Projects'].max()
clients['value_score'] = (clients['rev_norm']*0.65 + clients['proj_norm']*0.35)*100
top10 = clients.sort_values('value_score', ascending=False).head(10)

print('TOP 10 MOST VALUABLE CLIENTS:')
print(top10[['Client Name','City','Industry','Total Projects','Total Revenue (INR)','is_churned','value_score']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Top 10 Most Valuable Clients', fontsize=15, fontweight='bold')

churn_cols = ['#C62828' if c else '#1B5E20' for c in top10['is_churned']]
bars = axes[0].barh(top10['Client Name'], top10['Total Revenue (INR)']/1e6, color=churn_cols, edgecolor='white')
for bar, val, sc in zip(bars, top10['Total Revenue (INR)'], top10['value_score']):
    axes[0].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                 f'Rs.{val/1e6:.2f}M | Score:{sc:.0f}', va='center', fontsize=8.5)
axes[0].set_xlabel('Lifetime Revenue (Rs. Millions)')
axes[0].set_title('CLV (Green=Active, Red=Churned)', fontsize=11)
axes[0].set_xlim(0, top10['Total Revenue (INR)'].max()/1e6*1.52)

axes[1].barh(top10['Client Name'], top10['Total Projects'], color='#1565C0', edgecolor='white', alpha=0.85)
for bar, val in zip(axes[1].patches, top10['Total Projects']):
    axes[1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Total Projects')
axes[1].set_title('Total Projects per Top Client', fontsize=12)
plt.tight_layout()
plt.savefig('top10_clients.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. City-wise Client Distribution Map (Folium)
Run this cell to generate `client_city_map.html` — open it in any browser for an interactive map.

In [ ]:
CITY_COORDS = {
    'Delhi':(28.6139,77.2090),'Mumbai':(19.0760,72.8777),'Bengaluru':(12.9716,77.5946),
    'Hyderabad':(17.3850,78.4867),'Chennai':(13.0827,80.2707),'Kolkata':(22.5726,88.3639),
    'Pune':(18.5204,73.8567),'Ahmedabad':(23.0225,72.5714),'Jaipur':(26.9124,75.7873),
    'Surat':(21.1702,72.8311),'Lucknow':(26.8467,80.9462),'Chandigarh':(30.7333,76.7794),
    'Gurugram':(28.4595,77.0266),'Noida':(28.5355,77.3910),'Kochi':(9.9312,76.2673),
}

city_stats = clients.groupby('City').agg(
    client_count=('Client Name','count'),
    total_revenue=('Total Revenue (INR)','sum'),
    churned=('is_churned','sum'),
).reset_index()
city_stats['active'] = city_stats['client_count'] - city_stats['churned']
city_stats['lat']    = city_stats['City'].map(lambda c: CITY_COORDS.get(c,(0,0))[0])
city_stats['lon']    = city_stats['City'].map(lambda c: CITY_COORDS.get(c,(0,0))[1])

m = folium.Map(location=[20.5937,78.9629], zoom_start=5, tiles='CartoDB positron')
for _, row in city_stats.iterrows():
    if row['lat'] == 0: continue
    color = '#C62828' if row['churned'] > row['active'] else '#1B5E20'
    popup_html = (f"<b>{row['City']}</b><br>Clients: {row['client_count']}<br>"
                  f"Active: {row['active']} | Churned: {row['churned']}<br>"
                  f"Revenue: Rs.{row['total_revenue']/1e6:.2f}M")
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8+row['client_count']*4,
        color='white', weight=1.5, fill=True, fill_color=color, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=200),
        tooltip=f"{row['City']}: {row['client_count']} clients | Rs.{row['total_revenue']/1e6:.1f}M"
    ).add_to(m)
    folium.Marker(
        location=[row['lat']+0.4, row['lon']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;font-weight:bold;color:#222">{row["City"]}</div>',
            icon_size=(80,20), icon_anchor=(40,0)))\
    .add_to(m)

m.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;'
    'padding:10px 14px;border-radius:8px;border:1px solid #ccc;font-size:12px;">'
    '<b>YoungXCode Client Map</b><br>'
    '<span style="color:#1B5E20">&#9679;</span> Mostly Active<br>'
    '<span style="color:#C62828">&#9679;</span> Mostly Churned<br>'
    'Circle size = client count</div>'
))
m.save('client_city_map.html')
print('Saved: client_city_map.html — open in browser.')
print(city_stats[['City','client_count','active','churned','total_revenue']].sort_values('client_count',ascending=False).to_string(index=False))


## 9. City-wise & Industry-wise Breakdown Charts

In [ ]:
ind_stats = clients.groupby('Industry').agg(
    clients=('Client Name','count'),
    total_revenue=('Total Revenue (INR)','sum'),
    churned=('is_churned','sum'),
    avg_projects=('Total Projects','mean'),
).reset_index().sort_values('total_revenue', ascending=False)

print('INDUSTRY-WISE BREAKDOWN:')
print(ind_stats.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('City-wise & Industry-wise Client Analysis', fontsize=15, fontweight='bold')

cs = city_stats.sort_values('client_count', ascending=True)
bars = axes[0][0].barh(cs['City'], cs['client_count'], color=sns.color_palette('Blues_d',len(cs)), edgecolor='white')
for bar, val in zip(bars, cs['client_count']):
    axes[0][0].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=9, fontweight='bold')
axes[0][0].set_title('Clients by City', fontsize=12)

cs2 = city_stats.sort_values('total_revenue', ascending=True)
bars2 = axes[0][1].barh(cs2['City'], cs2['total_revenue']/1e6, color=sns.color_palette('Greens_d',len(cs2)), edgecolor='white')
for bar, val in zip(bars2, cs2['total_revenue']):
    axes[0][1].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2, f'Rs.{val/1e6:.1f}M', va='center', fontsize=8.5)
axes[0][1].set_xlabel('Revenue (Rs. Millions)')
axes[0][1].set_title('Revenue by City', fontsize=12)

pal = sns.color_palette('tab10', len(ind_stats))
wedges, texts, autotexts = axes[1][0].pie(ind_stats['clients'], labels=ind_stats['Industry'],
    autopct='%1.0f%%', colors=pal, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5), pctdistance=0.78)
for at in autotexts: at.set_fontsize(8); at.set_fontweight('bold')
axes[1][0].set_title('Client Share by Industry', fontsize=12)

ind2 = ind_stats.sort_values('total_revenue', ascending=True)
bars3 = axes[1][1].barh(ind2['Industry'], ind2['total_revenue']/1e6, color=sns.color_palette('viridis',len(ind2)), edgecolor='white')
for bar, val in zip(bars3, ind2['total_revenue']):
    axes[1][1].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2, f'Rs.{val/1e6:.1f}M', va='center', fontsize=9)
axes[1][1].set_xlabel('Revenue (Rs. Millions)')
axes[1][1].set_title('Revenue by Industry', fontsize=12)

plt.tight_layout()
plt.savefig('city_industry_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Re-engagement Opportunities Report

In [ ]:
re_engage = churned.copy()
re_engage['re_score'] = (
    re_engage['Total Revenue (INR)'] / re_engage['Total Revenue (INR)'].max() * 60 +
    re_engage['Total Projects']       / re_engage['Total Projects'].max()       * 40
).round(1)
re_engage = re_engage.sort_values('re_score', ascending=False)

print('RE-ENGAGEMENT PRIORITY REPORT:')
print('='*78)
print(f'  {"Client":<26} {"City":<13} {"Industry":<16} {"Projs":>6} {"Revenue":>14} {"Score":>7}')
print('-'*78)
for _, r in re_engage.iterrows():
    print(f'  {r["Client Name"]:<26} {r["City"]:<13} {r["Industry"]:<16} '
          f'{r["Total Projects"]:>6.0f} {r["Total Revenue (INR)"]:>14,.0f} {r["re_score"]:>7.1f}')
print('='*78)

fig, ax = plt.subplots(figsize=(13, 6))
re_cols = sns.color_palette('YlOrRd', len(re_engage))
bars_r  = ax.barh(re_engage['Client Name'], re_engage['re_score'], color=re_cols, edgecolor='white')
for bar, (_, r) in zip(bars_r, re_engage.iterrows()):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f"{r['Industry']} | Rs.{r['Total Revenue (INR)']/1e6:.1f}M | {r['Total Projects']:.0f} projs",
            va='center', fontsize=8.5)
ax.set_xlabel('Re-engagement Priority Score')
ax.set_title('Churned Clients — Re-engagement Priority Ranking', fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)
plt.tight_layout()
plt.savefig('re_engagement.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Key Findings & Recommendations

In [ ]:
best_ind    = ind_stats.iloc[0]['Industry']
best_city   = city_stats.sort_values('total_revenue', ascending=False).iloc[0]['City']
top_client  = top10.iloc[0]['Client Name']
top_target  = re_engage.iloc[0]['Client Name']
lost_rev    = churned['Total Revenue (INR)'].sum()

print('='*62)
print('   KEY FINDINGS')
print('='*62)
print(f'  1. Retention Rate          : {retention_rate:.1f}% ({repeat_clients}/{total_clients} repeat clients)')
print(f'  2. Churn Rate              : {churn_rate:.1f}% ({churned_count} inactive clients)')
print(f'  3. Avg Client LTV          : Rs.{avg_clv:,.0f}')
print(f'  4. Total Portfolio Revenue : Rs.{total_revenue:,.0f}')
print(f'  5. Revenue at Risk (churned): Rs.{lost_rev:,.0f}')
print(f'  6. Top Revenue City        : {best_city}')
print(f'  7. Top Revenue Industry    : {best_ind}')
print(f'  8. Most Valuable Client    : {top_client} (Score: {top10.iloc[0]["value_score"]:.0f})')
print(f'  9. Top Re-engage Target    : {top_target}')
print(f' 10. One-project at-risk     : {one_proj_clients} clients ({one_proj_clients/total_clients*100:.0f}%)')
print('='*62)
print()
print('='*62)
print('   ACTIONABLE RECOMMENDATIONS')
print('='*62)
print()
print('REC 1 — Upsell One-Project Clients (Highest Churn Risk)')
print('-'*62)
print(f'  {one_proj_clients} clients have only 1 project.')
print('  Launch 30-60-90 day post-delivery follow-ups offering')
print('  Phase 2 work, maintenance retainers, or audits.')
print()
print('REC 2 — Re-engage Churned Clients')
print('-'*62)
print(f'  Rs.{lost_rev/1e6:.1f}M in revenue is at risk from {churned_count} churned clients.')
print(f'  Priority: {top_target} and top 5 scored targets.')
print('  Use personalised proposals, case studies, and discounts.')
print()
print('REC 3 — Double Down on High-Performing Industries')
print('-'*62)
print(f'  {best_ind} leads in revenue. Allocate dedicated sales')
print('  resources and create industry-specific service packages.')
print()
print('REC 4 — Expand in High-Revenue Cities')
print('-'*62)
print(f'  {best_city} leads in revenue. Explore local partnerships')
print('  or account managers there. Target low-penetration cities')
print('  (Surat, Bengaluru) for new acquisition drives.')
print()
print('REC 5 — Launch Client Loyalty Program')
print('-'*62)
print('  Reward 5+ project clients with priority support and discounts.')
print(f'  Target: increase avg projects/client from {avg_projects:.1f} to 9+.')
print()
print('REC 6 — Monthly CLV Review in Business Meetings')
print('-'*62)
print(f'  Current avg CLV: Rs.{avg_clv:,.0f}.')
print('  Set 15% quarterly CLV growth target.')
print('  Prioritise account management effort by CLV tier.')
print('='*62)


## 12. Export Results

In [ ]:
clients.to_csv('client_full_analysis.csv', index=False)
churned[['Client Name','City','Industry','Total Projects','Total Revenue (INR)','days_since_last']]\
       .to_csv('churned_clients.csv', index=False)
top10[['Client Name','City','Industry','Total Projects','Total Revenue (INR)','value_score']]\
     .to_csv('top10_clients.csv', index=False)
re_engage[['Client Name','City','Industry','Total Projects','Total Revenue (INR)','re_score']]\
          .to_csv('reengagement_targets.csv', index=False)
ind_stats.to_csv('industry_breakdown.csv', index=False)
city_stats[['City','client_count','active','churned','total_revenue']]\
           .to_csv('city_breakdown.csv', index=False)

print('Exported:')
print('  -> client_full_analysis.csv')
print('  -> churned_clients.csv')
print('  -> top10_clients.csv')
print('  -> reengagement_targets.csv')
print('  -> industry_breakdown.csv')
print('  -> city_breakdown.csv')
print('  -> client_city_map.html  (open in browser)')
